# Kueue

A comprehensive guide to Kueue for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

Kueue is a **Kubernetes-native job queueing system** that manages the admission of batch workloads (Jobs, JobSets, etc.) into a cluster.

### What is it?

- A controller and set of CRDs that provide **queueing, quota management, and workload admission**.  
- Works alongside kube-scheduler, autoscalers, and existing job controllers.  
- Designed to support **multi-tenant, quota-aware batch clusters**.

### Why use it?

Key benefits of using Kueue:

- **Centralized job queueing**: Control when workloads start, rather than letting every Job compete for resources at once.  
- **Quota-aware scheduling**: Respect per-team or per-project resource quotas via ClusterQueues and LocalQueues.  
- **Separation of concerns**: Kueue handles queueing/admission, while kube-scheduler still decides pod-to-node placement.

### When to use it?

Kueue is particularly useful when:

- You run many **batch and ML jobs** on a shared Kubernetes cluster.  
- You need to enforce **fairness and quotas** across teams or projects.  
- You want to integrate with **JobSet or training operators** to coordinate large-scale AI workloads.

## Key Features

### Core Capabilities of Kueue

| Feature | Description | Benefit |
|--------|-------------|---------|
| **ClusterQueues** | Global queues associated with resource quotas (CPU, memory, GPUs). | Enforce fair usage across tenants. |
| **LocalQueues** | Namespaced queues that map to ClusterQueues. | Let teams submit jobs to their own queues. |
| **Workload CRD** | Internal abstraction that represents an admitted/queued job. | Decouples queueing from job controllers. |
| **Preemption & priorities** | Prioritize important workloads and preempt lower-priority ones. | Ensure SLAs for critical jobs. |
| **Integration with Jobs/JobSets** | Supports standard Jobs and advanced controllers (e.g., JobSet). | Incrementally adopt Kueue with existing workloads. |

## Architecture Overview

Kueue sits between **job submission** and **pod scheduling**.

```text
+-----------------------------+
|    Users / Job controllers  |
| (Jobs, JobSets, TFJob,...)  |
+---------------+-------------+
                |
                v
+-----------------------------+
|          Kueue              |
|  • ClusterQueue             |
|  • LocalQueue               |
|  • Workload CRD             |
+---------------+-------------+
                |
                v
+-----------------------------+
|  kube-scheduler & cluster   |
|  (pods, nodes, autoscaler)  |
+-----------------------------+
```

Kueue **does not replace** kube-scheduler; it controls when jobs become eligible for scheduling.

## Installation

Kueue is installed as a set of CRDs and controllers on your Kubernetes cluster.

Cluster/platform teams typically:

- Install Kueue via manifests or Helm.  
- Configure initial ClusterQueues, quotas, and policies.

As an ML practitioner, you interact by:

- Targeting the appropriate **LocalQueue** in your Jobs/JobSets.  
- Understanding quotas and priorities configured by the platform team.

In [ ]:
# Kueue is installed and configured cluster-wide, not via pip.

print("Once Kueue is installed, you use labels/annotations and queue references in your Job specs.")

## Basic Usage

### Example: Running a Job with Kueue

A typical pattern is:

1. Define a **ClusterQueue** with quotas (platform admin).  
2. Bind one or more **LocalQueues** in namespaces to that ClusterQueue.  
3. Submit a **Job** or **JobSet** that references the LocalQueue.

Below is a conceptual example of a LocalQueue and a Job using it.

In [ ]:
# Example LocalQueue and Job (YAML, conceptual)

kueue_yaml = """
apiVersion: kueue.x-k8s.io/v1beta1
kind: LocalQueue
metadata:
  name: team-a-queue
  namespace: team-a
spec:
  clusterQueue: ml-gpu-cluster-queue
---
apiVersion: batch/v1
kind: Job
metadata:
  name: team-a-training-job
  namespace: team-a
  labels:
    kueue.x-k8s.io/queue-name: team-a-queue
spec:
  template:
    spec:
      restartPolicy: Never
      containers:
      - name: trainer
        image: your-registry/trainer:latest
        resources:
          requests:
            cpu: "4"
            memory: "16Gi"
            nvidia.com/gpu: 1
"""

print(kueue_yaml)

# Kueue will queue and admit the Job based on ClusterQueue quotas and policies.

## Advanced Features

- **WorkloadPriorityClass**: Separate workload queuing priority from pod priority.  
- **Preemption**: Preempt lower-priority workloads when higher-priority work enters the queue.  
- **Integration with JobSet**: Manage multi-Job workloads under a single queueing and quota policy.  
- **Multiple ClusterQueues**: Different resource pools for different teams, environments, or cost profiles.

In [ ]:
# Placeholder for advanced Kueue YAML examples

print("See the Kueue docs for examples using WorkloadPriorityClass and JobSets.")

## Use Cases

- **Multi-tenant ML clusters**: Enforce per-team quotas and priorities for training and batch inference.  
- **Shared GPU clusters**: Queue GPU-intensive jobs to avoid oversubscription and thrashing.  
- **HPC/batch platforms on Kubernetes**: Combine Kueue with JobSet, Volcano, or training operators for robust batch scheduling.

## Best Practices

1. **Model quotas carefully**  
   - Set ClusterQueue quotas that reflect real capacity and business priorities.

2. **Use descriptive queue names and labels**  
   - Make it easy for users to find and target the right queues.

3. **Document queue semantics**  
   - Communicate expectations around wait times, priorities, and preemption to ML teams.

4. **Start with a pilot**  
   - Roll out Kueue gradually to avoid surprising users with new queueing behavior.

5. **Monitor admission rates and wait times**  
   - Tune quotas and priorities based on observed patterns.

## Common Pitfalls

1. **Misaligned quotas**  
   - Symptom: Jobs waiting too long or being admitted in bursts.  
   - Fix: Adjust ClusterQueue quotas and scaling policies.

2. **Confusion between pod priority and workload priority**  
   - Symptom: Unexpected preemptions or ordering.  
   - Fix: Use WorkloadPriorityClass appropriately and document behavior.

3. **Ignoring observability**  
   - Symptom: Difficult to explain why some jobs are waiting.  
   - Fix: Expose queue metrics and use dashboards for transparency.

## Performance Optimization

- **Match ClusterQueues to physical capacity**:  
  - Avoid overcommitting quotas far beyond what the cluster can actually run.

- **Use priority and preemption sparingly**:  
  - Too much preemption can hurt overall throughput; tune carefully.

- **Align autoscaling with Kueue**:  
  - Ensure cluster-autoscaler and node groups can respond to queued demand in a timely way.

In [ ]:
# Placeholder for metrics/monitoring examples

print("Use Kueue-provided metrics (e.g., workload counts, admission latencies)\n"
      "alongside cluster metrics to tune performance.")

## Production Deployment

- **Treat Kueue as core cluster infrastructure**:  
  - Managed and upgraded by platform teams, with clear SLOs.

- **Multi-cluster strategies**:  
  - Use separate Kueue instances per cluster or region; design around failover scenarios.

- **Governance**:  
  - Align queue priorities and quotas with organizational policies and budgets.

## Monitoring and Observability

- **Metrics**:  
  - Expose Kueue-specific metrics for workloads, queues, and admit/reject decisions.  

- **Dashboards**:  
  - Provide Grafana dashboards for job wait times, queue utilization, and quota usage.

- **Logging & events**:  
  - Use Kubernetes events and controller logs to debug admission behavior.

## Troubleshooting

- **Jobs never start**:  
  - Check LocalQueue and ClusterQueue configuration; look for quota or policy issues.

- **Unexpected preemption**:  
  - Review priorities and policies; inspect events for preemption reasons.

- **Workload CRDs stuck**:  
  - Investigate controller logs; confirm compatibility with your Kubernetes version.

## Comparison with Alternatives

| Aspect | Kueue | Native Kubernetes | Vendor schedulers (Run:ai, etc.) |
|--------|-------|-------------------|-----------------------------------|
| Type | OSS queueing controller | Baseline job scheduling | Commercial platforms |
| Focus | Quota-aware job admission | Pod scheduling | GPU utilization, dashboards, enterprise features |
| Best for | Shared, multi-tenant K8s clusters | Simple or small clusters | Enterprises wanting managed solutions |

Choose Kueue when you:

- Need **open-source, Kubernetes-native** queueing and quota management.  
- Want to coordinate many batch/ML jobs on shared clusters with clear fairness policies.

## Resources

- Kueue homepage: https://kueue.sigs.k8s.io/  
- Overview: https://kueue.sigs.k8s.io/docs/overview/  
- Tasks & examples: https://kueue.sigs.k8s.io/docs/tasks/run/jobs/

These resources show how to configure ClusterQueues, LocalQueues, and run Jobs/JobSets with Kueue.